# Milvus + Woodpecker: full-stack test walkthrough

Run this notebook from top to bottom with **Shift+Enter**, or use **Run All** after reviewing the configuration.
It invokes the existing `stack.sh` commands and uses the same dashboards, alerts and performance runner as the CLI.

| Stage | What you will see |
| --- | --- |
| Prepare and deploy | Target profile/context, rendered charts and ready Pods |
| Open endpoints | Four E2E dashboards, three reference dashboards, Prometheus and Alertmanager |
| Write smoke | A new collection, inserts, primary-key query and vector search |
| Verify monitoring | Scrape health, all dashboard queries and alert coverage gaps |
| Verify alert pipeline | Offline rule tests, firing, AM delivery, recovery and inhibition |
| Run write performance | Throughput, p50/p95/p99, correctness, latency plot, metrics and logs |
| Diagnose and collect | Slow requests, timestamped Pod logs and machine-readable evidence |
| Pause or reset | Separate opt-in cells for stop/start and clear/deploy |

Docker Desktop must be running. Install minikube, Helm 3, kubectl and curl first; the notebook runtime uses Python 3.11+.
Allow several minutes for deployment, scrape windows, rule reloads and performance collection.
Email is disabled. Alerts are visible in Prometheus and Alertmanager.

**Run All leaves the stack running.** Stop/clear cells are disabled by default. `clear` deletes this stack's persistent data.

In [ ]:
from pathlib import Path
import html
import json
import os
import shutil
import subprocess
import sys
import time
from IPython.display import HTML, Markdown, display

candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend([parent, parent / "deployments" / "milvus-full-stack-helm"])
ROOT = next((p for p in candidates if (p / "stack.sh").is_file() and (p / "scripts" / "perf.py").is_file()), None)
assert ROOT is not None, "Open this notebook from inside the repository"
print("Stack directory:", ROOT)

## 1. Choose the target and load

The defaults match the supplied local ARM64 build. Set `USE_LOCAL_ARM64=False` for the public image/default x86 deployment;
image compatibility and registry access still need to match your platform. The private ARM64 image must be available locally or through your registry credentials.

`RUN_IMAGES=True` explicitly preloads the required images. Set it to `False` only when all images are already loaded in the current minikube node.
For an existing external Kubernetes cluster, set its context, set `RUN_START=False`, and keep `RUN_IMAGES=False`.

All four ports must belong to this stack. Existing healthy endpoints are reused; the notebook does not stop external forward processes.
Choose four different free ports if other applications or a different stack use the defaults.

In [ ]:
PROFILE = os.getenv("PROFILE", "wp-ops-v0146")
CONTEXT = os.getenv("CONTEXT", PROFILE)
NAMESPACE = os.getenv("NAMESPACE", "milvus-stack")
PORTS = {key: int(os.getenv(key + "_PORT", default)) for key, default in {
    "GRAFANA": 3000, "PROMETHEUS": 9099, "MILVUS": 19530, "ALERTMANAGER": 9093}.items()}
USE_LOCAL_ARM64 = True
RUN_START = True
RUN_IMAGES = True
RUN_DEPLOY = True
RUN_RULE_TESTS = True
RUN_ALERT_PROBE = True
RUN_PERF = True
RUN_GATE_DEMO = False
RUN_ACCEPTANCE = False  # Repeats the earlier tests; useful as a single CI command.

SMOKE_SETTINGS = {"BATCHES": "3", "BATCH_SIZE": "100", "INTERVAL": "2"}
PERF_SETTINGS = {"PERF_DURATION": "60", "PERF_WARMUP": "5", "PERF_COOLDOWN": "35",
                 "PERF_CONCURRENCY": "2", "PERF_BATCH_SIZE": "100", "PERF_DIMENSION": "128",
                 "PERF_RPS": "5", "PERF_MIN_ROWS_PER_SECOND": "0", "PERF_MAX_P99_MS": "0"}

In [ ]:
if "lab" in globals():
    lab.close_forwards()  # Only closes forwards that this notebook created.
sys.path.insert(0, str(ROOT / "scripts"))
from notebook_helpers import StackLab, show_report
lab = StackLab(ROOT, PROFILE, CONTEXT, NAMESPACE, PORTS, USE_LOCAL_ARM64)
print(json.dumps({"profile": PROFILE, "context": CONTEXT, "namespace": NAMESPACE,
                  "ports": PORTS, "local_arm64": USE_LOCAL_ARM64}, indent=2))
required = ["docker", "helm", "kubectl", "curl"] + (["minikube"] if RUN_START or RUN_IMAGES else [])
missing = [name for name in required if not shutil.which(name)]
assert not missing, f"Install these tools first: {missing}"
print("Required tools found. Notebook commands stream output and fail on nonzero exit status.")
lab.run("help")

## 2. Start, prepare and deploy

`start` creates/resumes the selected local minikube profile. `deploy` installs/upgrades the three Helm releases and preserves existing PVC data.
The render output lives in `.local/rendered/`. Re-run these cells after changing values; keep the same image/profile settings.

Each command also saves its streamed output to `.local/notebook/<timestamp>-<command>.log`, including failed startup or deployment attempts.


In [ ]:
if RUN_START:
    lab.run("start")
else:
    print("Using the configured existing Kubernetes context:", CONTEXT)

In [ ]:
lab.run("render")
if RUN_IMAGES:
    lab.run("images")
else:
    print("Image preload skipped. Use RUN_IMAGES=True for a first local preload.")

In [ ]:
if RUN_DEPLOY:
    lab.run("deploy")
lab.run("status")

## 3. Open dashboards and services

The following cell reuses all four existing healthy endpoints or starts background forwards owned by this notebook.
It waits for readiness and does not block the kernel indefinitely. Forward output is in `.local/notebook/forward.log`.
If only some ports are occupied, it stops and asks you to fix the port configuration.

Grafana login: **admin / admin**. Select **Last 15 minutes** and a **5-second refresh** for live observation.
Keep the notebook kernel and its forwards running while using the dashboards.

In [ ]:
lab.open_forwards()
grafana = lab.env["GRAFANA_URL"]
links = {"E2E Write / Append": grafana + "/d/milvus-e2e-write",
         "E2E WAL Read / Consume": grafana + "/d/milvus-e2e-consume",
         "E2E Flush / Checkpoint": grafana + "/d/milvus-e2e-flush",
         "E2E Search / Query": grafana + "/d/milvus-e2e-query",
         "Milvus dashboard": grafana + "/d/milvus-full-stack",
         "Woodpecker client dashboard": grafana + "/d/woodpecker-client-k8s",
         "Woodpecker server dashboard": grafana + "/d/woodpecker-server-k8s",
         "Prometheus alerts": lab.env["PROMETHEUS_URL"] + "/alerts",
         "Alertmanager": lab.env["ALERTMANAGER_URL"], "Milvus REST": lab.env["MILVUS_URL"]}
display(HTML("<table><tr><th>Service</th><th>Local URL</th></tr>" + "".join(
    '<tr><td>' + html.escape(name) + '</td><td><a target="_blank" href="' + html.escape(url, quote=True) + '">' + html.escape(url) + '</a></td></tr>'
    for name, url in links.items()) + "</table>"))

## 4. Insert a small workload and verify it

The default smoke inserts **300 eight-dimensional vectors** in three batches.
Every batch checks an ID query and nearest-vector search. A uniquely named collection is retained for inspection.
The final event should say `stage: passed`. Results are in `.local/workload.jsonl`.

In [ ]:
lab.run("smoke", env=SMOKE_SETTINGS)
smoke_events = [json.loads(line) for line in (lab.state / "workload.jsonl").read_text().splitlines()]
assert smoke_events[-1]["stage"] == "passed"
display(smoke_events[-1])

## 5. Check collection, dashboards and alert coverage

Wait for two 15-second scrapes after writing, then check all seven dashboards, their PromQL expressions and required E2E stage metrics.
Queries with no samples are recorded separately from query errors: this workload does not exercise every feature or error counter.
The alert report distinguishes rule evaluation health from missing metric/history coverage. C2 needs seven-day history; C6 read-frontier instrumentation may be absent in the chosen image.
The readiness barrier waits for eight business targets and all reference rules to be healthy on two consecutive checks. It tolerates brief scrape/rollout transitions, but raises an error after 180 seconds if they persist.


In [ ]:
print("Waiting 35 seconds for the final writes to reach Prometheus...")
time.sleep(35)
lab.wait_monitoring(timeout=180)
lab.run("verify")
verification = json.loads((lab.state / "verification.json").read_text())
display({uid: {key: value for key, value in report.items() if key in ("queries", "queries_with_data", "errors")}
         for uid, report in verification["dashboards"].items()})

In [ ]:
lab.run("alerts")
alert_report = json.loads((lab.state / "alerting.json").read_text())
gaps = [{key: row.get(key) for key in ("rule_id", "missing_metric_names", "seven_day_history_available")}
        for row in alert_report["rules"] if row["missing_metric_names"] or row.get("seven_day_history_available") is False]
display({"notification_mode": alert_report["notifications"], "coverage_gaps": gaps})

## 5a. Inspect real request traces

Jaeger is available at `http://127.0.0.1:16686` (or `JAEGER_PORT`). Grafana Explore also has a provisioned **Jaeger** datasource. The shared Milvus/WP configuration exports OTLP with full sampling for small local diagnostic runs. Read [TRACING.md](../TRACING.md) before sustained performance comparisons.

Enable the following optional 100-vector probe to save exact request trace IDs and raw spans. The supplied image currently shows a **partial** Insert trace: Proxy/coordinator spans are exported, but SN/WP do not remain under that user request ID. Query/search traces include SN and segcore. Separate WP traces do not prove a connected write chain.

Use `TRACES_REQUIRE_E2E=1 ./stack.sh traces-smoke` for a strict automated check that fails on these gaps. Jaeger history is bounded and in memory; save JSON evidence before eviction or restart.


In [ ]:
RUN_TRACE_SMOKE = False
if RUN_TRACE_SMOKE:
    lab.run("traces-smoke")
    trace_report = json.loads((lab.state / "tracing-smoke.json").read_text())
    display({key: trace_report.get(key) for key in ("status", "services", "coverage_gaps", "failures")})
    for request in trace_report.get("requests", []):
        trace = request.get("trace", {})
        if trace.get("url"):
            display(HTML('<a target="_blank" href="' + html.escape(trace["url"]) + '">'
                         + html.escape(request["operation"]) + '</a>'))
else:
    print("Optional trace probe skipped. Set RUN_TRACE_SMOKE=True to insert 100 vectors and inspect coverage.")


## 6. Exercise the alert pipeline

The offline tests check 16 alert behaviors with the pinned Prometheus image.
The live probe creates temporary synthetic rules/events, verifies firing → AM receipt → recovery and inhibition, and cleans up afterward.
It takes several minutes because Kubernetes rule files reload asynchronously. You can watch Prometheus and AM while it runs.
It does not stop business Pods or send email. External notification configurations are refused by this local smoke probe.

In [ ]:
if RUN_RULE_TESTS:
    lab.run("alerts-test")
else:
    print("Offline rule tests skipped.")

In [ ]:
if RUN_ALERT_PROBE:
    lab.run("alerts-smoke")
    probe = json.loads((lab.state / "alerting-smoke.json").read_text())
    display({key: probe.get(key) for key in ("pipeline_probe", "inhibition_probe", "failures")})
else:
    print("Live alert probe skipped.")

## 7. Measure the real write path

Default measurement: 60 seconds, 2 workers, 100 rows/batch, 128 dimensions and a total target of 5 requests/s (500 rows/s).
Warm-up statistics are separate. This is a local REST/port-forward baseline, not a maximum-capacity result.
Change `PERF_SETTINGS` above to repeat the experiment; use hardware-specific throughput/latency gates when you have a baseline.

After writing, the runner checks strong-read counts, sampled IDs and filtered vector search. It collects metrics, AM snapshots and 10 business/dependency Pod logs even on handled failures.
These checks establish live visibility; they do not test crash/recovery durability. WP metrics aggregate the deployment rather than tracing each request end to end.

In [ ]:
perf_path = None
perf_report = None
if RUN_PERF:
    perf_rc = lab.run("perf", env=PERF_SETTINGS, check=False)
    perf_path = lab.latest_perf()
    perf_report = show_report(perf_path)
    assert perf_rc == 0, f"Performance run failed. Inspect {perf_path} and the evidence cells below."
else:
    print("Performance run skipped. Existing reports remain under .local/perf/.")

## 8. Inspect the evidence and slowest requests

Each `.local/perf/<run-id>/` directory includes `summary.md`, `report.json`, every insert attempt in `requests.jsonl`, metric ranges, snapshots, Pod status/events and bounded timestamped logs.
The report has dashboard links for the test time window. Local request IDs are not assumed to be server trace IDs; correlate collection metadata, timestamps, log identity and node labels.
Missing lazy error counters are coverage gaps. An empty `ALERTS` series can simply mean no pending/firing rules.

In [ ]:
if perf_path:
    display({"report": str(perf_path), "collection_info": perf_report.get("collection_info"),
             "correctness": perf_report.get("correctness"), "triage": perf_report.get("triage"),
             "diagnostic_errors": perf_report["diagnostic_errors"]})
    requests = [json.loads(line) for line in (perf_path.parent / "requests.jsonl").read_text().splitlines()]
    slowest = sorted(requests, key=lambda row: row["latency_ms"], reverse=True)[:5]
    display([{key: row.get(key) for key in ("request_id", "at", "phase", "latency_ms", "ok", "error")} for row in slowest])
    display({name: {key: row.get(key) for key in ("finite_samples", "max", "counter_delta")}
             for name, row in perf_report["metrics"].items()})
else:
    print("Run perf first to create a correlated experiment directory.")

In [ ]:
if perf_path:
    log_files = sorted(perf_path.parent.glob("*.log"))
    print("Log files:", [path.name for path in log_files])
    current_logs = [path for path in log_files if not path.name.endswith("-previous.log")]
    preferred = [path for path in current_logs if "woodpecker-0" in path.name]
    selected = next((path for path in preferred + current_logs if path.stat().st_size > 0), None)
    if selected:
        print("Last 20 lines:", selected.name)
        tail = "\n".join(selected.read_text(errors="replace").splitlines()[-20:])[-12000:]
        display(HTML("<pre>" + html.escape(tail) + "</pre>"))

## 9. Optional: demonstrate a failed performance gate

Set `RUN_GATE_DEMO=True` to write 1,000 additional rows at a small rate with an intentionally impossible 0.001-ms p99 gate.
The expected outcome is a nonzero status with successful writes/read checks and retained diagnostic evidence.
This demonstration does not inject a failure into Milvus or Woodpecker.

In [ ]:
if RUN_GATE_DEMO:
    gate_rc = lab.run("perf", env={**PERF_SETTINGS, "PERF_DURATION": "5", "PERF_WARMUP": "0",
                                  "PERF_RPS": "2", "PERF_MAX_P99_MS": "0.001"}, check=False)
    gate_path = lab.latest_perf()
    gate_report = show_report(gate_path)
    assert gate_rc != 0 and any("p99" in failure for failure in gate_report["failures"])
else:
    print("Gate failure demonstration skipped. Set RUN_GATE_DEMO=True to exercise it.")

## 10. Collect wider diagnostics or repeat automated acceptance

`collect` includes all Pod logs, resource status, images and dashboard/alert reports under a timestamped `.local/evidence/` directory.
The optional `acceptance` command repeats the earlier rule/workload/dashboard/live-probe checks as one CLI entry point. It does not run perf or email tests by default.
Review application logs before sharing evidence.

In [ ]:
lab.run("collect")
if RUN_ACCEPTANCE:
    lab.run("acceptance")
else:
    print("Repeated acceptance skipped; CLI equivalent: ./stack.sh acceptance")
print("Evidence directory:", lab.state / "evidence")

## 11. Finish, pause/resume, or reset

The defaults below leave the stack and dashboard access running after Run All.

| Action | Effect |
| --- | --- |
| `lab.close_forwards()` | Close only this notebook's forwards; keep the stack and data |
| `stop` / `start` | Pause/resume local minikube; keep deployments and data; reopen forwards after resuming |
| `clear` / `deploy` | Delete this stack's releases and PVC data, then create a fresh deployment |

Run only the specific lifecycle cell you need. `clear` is destructive to this stack's stored collections, WAL, metadata and monitoring history.
It preserves minikube, the namespace, shared CRDs and local evidence. Shutting down a kernel is not a substitute for `stop`.

In [ ]:
CLOSE_NOTEBOOK_FORWARDS = False
if CLOSE_NOTEBOOK_FORWARDS:
    lab.close_forwards()
    print("Notebook-owned forwards closed. External forwards were left alone.")
else:
    print("Keeping dashboard access available.")

In [ ]:
STOP_CLUSTER = False
if STOP_CLUSTER:
    lab.close_forwards()
    lab.run("stop")
else:
    print("Cluster remains running. To pause it, set STOP_CLUSTER=True and run this cell.")

In [ ]:
RESUME_CLUSTER = False
if RESUME_CLUSTER:
    lab.run("start")
    lab.open_forwards()
else:
    print("Resume skipped. Set RESUME_CLUSTER=True when you want to restart a stopped local cluster.")

In [ ]:
CLEAR_STACK_AND_DATA = False
if CLEAR_STACK_AND_DATA:
    lab.close_forwards()
    lab.run("clear")
else:
    print("Clear skipped: all stack data is retained. Set CLEAR_STACK_AND_DATA=True only for a fresh reset.")

In [ ]:
REDEPLOY_AFTER_CLEAR = False
if REDEPLOY_AFTER_CLEAR:
    lab.run("deploy")
    lab.open_forwards()
else:
    print("Fresh redeployment skipped.")

## References

- [Stack README](../README.md): deployment, commands and dashboard sources.
- [Performance guide](../PERFORMANCE.md): load settings, gates, evidence schema and investigation workflow.
- [Alerting guide](../ALERTING.md): rule catalogue, coverage gaps and optional SMTP examples.

Notebook outputs are local observations. Clear all outputs before sharing/committing an edited source notebook;
the shipped notebook contains no executed output, credentials or machine-specific paths.